# Medical AI Chatbot


In [1]:
from IPython.display import clear_output
!pip install bitsandbytes accelerate gradio
import gradio as gr
clear_output()

In [3]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM


# MODEL NAME


model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# ==========================================
# DEVICE
# ==========================================

device = "mps" if torch.backends.mps.is_available() else "cpu"

print("Using device:", device)


# TOKENIZER

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

# MODEL

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "mps" else torch.float32
)

model.to(device)

model.eval()

print("Model loaded successfully!")


# CHAT FUNCTION

def chat(message, history):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful Medical AI Assistant. "
                "Provide safe medical guidance."
            )
        }
    ]

    # Add chat history
    for user_msg, bot_msg in history:

        messages.append(
            {
                "role": "user",
                "content": str(user_msg)
            }
        )

        messages.append(
            {
                "role": "assistant",
                "content": str(bot_msg)
            }
        )

    # Current user message
    messages.append(
        {
            "role": "user",
            "content": str(message)
        }
    )

    # Create prompt
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    # Generate
    with torch.no_grad():

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,

            max_new_tokens=128,

            temperature=0.7,
            top_p=0.9,

            do_sample=True,

            repetition_penalty=1.1,

            pad_token_id=tokenizer.eos_token_id
        )

    # Remove prompt tokens
    generated_tokens = outputs[0][input_ids.shape[1]:]

    # Decode
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


# GRADIO UI

demo = gr.ChatInterface(
    fn=chat,
    title="Medical AI Assistant",
    description="Ask your medical questions here."
)

# LAUNCH

demo.launch(
    share=True,
    inbrowser=True
)

Using device: mps
Model loaded successfully!
Running on local URL:  http://127.0.0.1:7860


/opt/anaconda3/lib/python3.12/site-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


Running on public URL: https://7b07517a57e687a848.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
